In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
 
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ildarlukmanov/cubicasa5k-yolo-formatted/data.yaml
/kaggle/input/datasets/ildarlukmanov/cubicasa5k-yolo-formatted/images/train/1.png
/kaggle/input/datasets/ildarlukmanov/cubicasa5k-yolo-formatted/images/train/10.png
/kaggle/input/datasets/ildarlukmanov/cubicasa5k-yolo-formatted/images/train/100.png
/kaggle/input/datasets/ildarlukmanov/cubicasa5k-yolo-formatted/images/train/1000.png
/kaggle/input/datasets/ildarlukmanov/cubicasa5k-yolo-formatted/images/train/1001.png
/kaggle/input/datasets/ildarlukmanov/cubicasa5k-yolo-formatted/images/train/1002.png
/kaggle/input/datasets/ildarlukmanov/cubicasa5k-yolo-formatted/images/train/1003.png
/kaggle/input/datasets/ildarlukmanov/cubicasa5k-yolo-formatted/images/train/1004.png
/kaggle/input/datasets/ildarlukmanov/cubicasa5k-yolo-formatted/images/train/1005.png
/kaggle/input/datasets/ildarlukmanov/cubicasa5k-yolo-formatted/images/train/1006.png
/kaggle/input/datasets/ildarlukmanov/cubicasa5k-yolo-formatted/images/train/1007.pn

# Router & Metadata Expert

## Architecture: ResNet-18 (Classification) + PaddleOCR (Text Extraction).

In [2]:
# ═════════════════════════════════════════════════════════════════════════════
# CELL 1: Setup & Dataset Generation (CORRECTED)
# ═════════════════════════════════════════════════════════════════════════════
import os, random, json
import yaml
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from tqdm import tqdm

WORK_DIR = Path("/kaggle/working")
DATASET_DIR = WORK_DIR / "router-expert-dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# Sheet numbering logic: [Discipline][Type][Sequence]
DISCIPLINES = ["G", "A", "AS", "S", "M", "E", "P", "C", "L"]

def generate_router_dataset(n=1000):
    # YOLO only needs 1 class: finding the sheet number text block.
    # PaddleOCR will handle reading the actual characters.
    classes = {0: "sheet_number"}
    
    for split, count in [("train", int(n*0.8)), ("val", int(n*0.2))]:
        img_dir = DATASET_DIR / split / "images"
        lbl_dir = DATASET_DIR / split / "labels"
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"🎨 Generating {split} split: {count} images...")
        for i in tqdm(range(count)):
            # Create a tall image simulating a title block area or full sheet bottom
            img = Image.new("RGB", (800, 1000), "white")
            draw = ImageDraw.Draw(img)
            
            # Draw fake title block borders
            draw.rectangle([50, 800, 750, 980], outline="black", width=3)
            draw.line([50, 850, 750, 850], fill="black", width=2)
            draw.line([400, 850, 400, 980], fill="black", width=2)
            
            # Generate sheet number (e.g., A101, S203)
            disc = random.choice(DISCIPLINES)
            type_num = random.choice(["0", "1", "2", "3", "4", "5"])
            seq = f"{random.randint(1, 99):02d}"
            sheet_num = f"{disc}{type_num}{seq}"
            
            # Randomly place the sheet number in the right-side box of the title block
            text_x = random.randint(450, 650)
            text_y = random.randint(870, 930)
            
            try:
                font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", random.randint(28, 42))
            except:
                font = ImageFont.load_default()
                
            draw.text((text_x, text_y), sheet_num, fill="black", font=font)
            draw.text((60, 870), "PROJECT: MOCK BUILDING", fill="black")
            draw.text((60, 920), "DRAWN BY: AI", fill="black")
            
            img.save(img_dir / f"{i:04d}.png")
            
            # Calculate YOLO bounding box for the sheet number
            bbox = font.getbbox(sheet_num)
            tw = bbox[2] - bbox[0]
            th = bbox[3] - bbox[1]
            
            # Add padding so OCR doesn't clip the edges of the letters
            pad = 20 
            x1 = max(0, text_x - pad)
            y1 = max(0, text_y - pad)
            x2 = min(800, text_x + tw + pad)
            y2 = min(1000, text_y + th + pad)
            
            # Convert to YOLO format (cx, cy, w, h normalized 0-1)
            cx = (x1 + x2) / 2 / 800
            cy = (y1 + y2) / 2 / 1000
            w = (x2 - x1) / 800
            h = (y2 - y1) / 1000
            
            with open(lbl_dir / f"{i:04d}.txt", "w") as f:
                f.write(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")

    # ── CRITICAL: Write the data.yaml file! ───────────────────────────────
    data_yaml = {
        "path": str(DATASET_DIR.resolve()),
        "train": "train/images",
        "val": "val/images",
        "nc": 1,
        "names": {0: "sheet_number"}
    }
    
    yaml_path = DATASET_DIR / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(data_yaml, f, default_flow_style=False)
        
    print(f"✅ Dataset and data.yaml successfully generated at {DATASET_DIR}")
    return yaml_path

# Run the generator
yaml_path = generate_router_dataset(n=1000)

🎨 Generating train split: 800 images...


100%|██████████| 800/800 [00:19<00:00, 41.11it/s]


🎨 Generating val split: 200 images...


100%|██████████| 200/200 [00:04<00:00, 41.46it/s]

✅ Dataset and data.yaml successfully generated at /kaggle/working/router-expert-dataset


In [3]:
# ═════════════════════════════════════════════════════════════════════════════
# CELL 2: Install Dependencies & Initialize OCR
# ═════════════════════════════════════════════════════════════════════════════
# Remove '==8.3.40' to let it install the latest patched version
!pip install -q ultralytics paddlepaddle paddleocr onnx onnxruntime opencv-python-headless

import cv2
import numpy as np
import json
import re
import shutil
from pathlib import Path
from ultralytics import YOLO
from paddleocr import PaddleOCR

# Initialize PaddleOCR (English, lightweight, CPU-friendly)
# Updated for newer PaddleOCR versions: 
# - Removed 'use_gpu' (no longer a valid top-level argument)
# - Replaced deprecated 'use_angle_cls' with 'use_textline_orientation'
try:
    ocr_engine = PaddleOCR(lang='en', use_textline_orientation=True)
except TypeError:
    # Fallback for slightly older versions that might still expect the old argument
    ocr_engine = PaddleOCR(lang='en', use_angle_cls=True)

print("✅ Dependencies installed and PaddleOCR initialized.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires PyYAML<6.1,>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('UVDoc', None, None)
Using official model (UVDoc), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/UVDoc`.


...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Creating model: ('PP-OCRv6_medium_rec', None, None)
Using official model (PP-OCRv6_medium_rec), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Dependencies installed and PaddleOCR initialized.


In [4]:
# ═════════════════════════════════════════════════════════════════════════════
# CELL 3: Train YOLOv8n for Sheet Number Detection (FIXED)
# ═════════════════════════════════════════════════════════════════════════════

# 🔥 CRITICAL FIX: Uninstall 'ray' to prevent the Ultralytics callback crash
!pip uninstall -y ray ray[tune]

print("🚀 Starting YOLOv8n training for sheet number detection...")
# Load the nano model for maximum speed in the routing phase
model = YOLO('yolov8n.pt')

# Train the model
results = model.train(
    data=str(DATASET_DIR / 'data.yaml'),
    epochs=50,             # Synthetic data converges quickly
    imgsz=640,
    batch=16,
    device=0,              # Use GPU if available
    project=str(WORK_DIR / 'runs'),
    name='router_expert',
    exist_ok=True,
    patience=10,
    plots=True,
    # Augmentations to simulate real scanned plans
    degrees=10.0,          # Title blocks are often rotated
    translate=0.1,
    scale=0.2,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.5
)

# Validate the trained model
val_results = model.val(data=str(DATASET_DIR / 'data.yaml'), split='val')
print(f"\n📊 Router Expert Validation Metrics:")
print(f"   mAP50:    {val_results.box.map50:.4f}")
print(f"   mAP50-95: {val_results.box.map:.4f}")

Found existing installation: ray 2.59.0
Uninstalling ray-2.59.0:
  Successfully uninstalled ray-2.59.0
🚀 Starting YOLOv8n training for sheet number detection...
Ultralytics 8.4.190 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/router-expert-dataset/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.5, hsv_v=0.5, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01

In [5]:
# ═════════════════════════════════════════════════════════════════════════════
# CELL 4: ONNX Export & Production Inference Wrapper (FULLY FIXED)
# ═════════════════════════════════════════════════════════════════════════════

print("📦 Exporting YOLO model to ONNX...")
best_model = YOLO(str(WORK_DIR / 'runs' / 'router_expert' / 'weights' / 'best.pt'))

# Export to ONNX with dynamic batch size and simplified graph
onnx_path = best_model.export(
    format='onnx', 
    imgsz=640, 
    dynamic=True, 
    simplify=True, 
    opset=17
)

# Copy to standardized output directory
OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
shutil.copy(onnx_path, OUTPUT_DIR / "router-expert.onnx")
print(f"✅ Model exported and saved to {OUTPUT_DIR / 'router-expert.onnx'}")

# ── Production Inference Wrapper ──────────────────────────────────────────────
class RouterExpert:
    """
    MOSAIC Router Expert: Detects sheet numbers and routes to the correct MoE domain.
    """
    def __init__(self, onnx_path: str, ocr_engine):
        import onnxruntime as ort
        # Note: CUDAExecutionProvider will gracefully fall back to CPU if no GPU is available
        self.session = ort.InferenceSession(onnx_path, providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
        self.ocr = ocr_engine
        self.input_name = self.session.get_inputs()[0].name
        
        # Regex to capture standard sheet numbering: [1-2 Letters][1 Digit][2 Digits]
        self.sheet_regex = re.compile(r'\b([A-Z]{1,2})(\d)(\d{2})\b')
        
        self.routing_map = {
            "G": "general", "A0": "general",
            "AS": "site_plan",
            "A1": "floor_plan", "A2": "floor_plan", 
            "A3": "section", 
            "A4": "detail_schedule", "A5": "detail_schedule",
            "S": "structural",
            "M": "mep", "E": "mep", "P": "mep",
            "C": "civil", "L": "landscape"
        }

    def preprocess(self, image: np.ndarray) -> np.ndarray:
        """Resize and normalize image for YOLO."""
        img = cv2.resize(image, (640, 640))
        img = img.transpose(2, 0, 1)[::-1] # BGR to RGB, HWC to CHW
        return np.ascontiguousarray(img / 255.0, dtype=np.float32)

    def predict(self, image):
        # 1. YOLO Detection
        input_tensor = self.preprocess(image)
        
        # ✅ FIX 1: Add batch dimension to make it (1, 3, 640, 640)
        input_tensor = np.expand_dims(input_tensor, axis=0)
        
        # Run ONNX inference
        outputs = self.session.run(None, {self.input_name: input_tensor})
        
        # Simple NMS parsing: assume the highest confidence detection is the sheet number
        # outputs[0] shape is (1, 84, 8400) for YOLOv8 with 1 class
        scores = outputs[0][0, 4, :]
        best_idx = np.argmax(scores)
        
        if scores[best_idx] < 0.5:
            return {"expert": "manual_review", "confidence": 0.0, "reason": "No sheet number detected"}

        # ✅ FIX 2: YOLOv8 ONNX outputs boxes in 640x640 scale, NOT normalized 0-1!
        bbox = outputs[0][0, :4, best_idx]
        h_img, w_img = image.shape[:2]
        
        # Scale coordinates from 640x640 back to original image dimensions
        scale_x = w_img / 640.0
        scale_y = h_img / 640.0
        
        cx = bbox[0] * scale_x
        cy = bbox[1] * scale_y
        w = bbox[2] * scale_x
        h = bbox[3] * scale_y
        
        x1 = int(max(0, cx - w/2))
        y1 = int(max(0, cy - h/2))
        x2 = int(min(w_img, cx + w/2))
        y2 = int(min(h_img, cy + h/2))

        # 2. Crop and Run PaddleOCR
        crop = image[y1:y2, x1:x2]
        ocr_result = self.ocr.ocr(crop, cls=True)
        
        extracted_text = ""
        ocr_confidence = 0.0
        if ocr_result and ocr_result[0]:
            for line in ocr_result[0]:
                text = line[1][0]
                conf = line[1][1]
                if conf > ocr_confidence:
                    extracted_text = text
                    ocr_confidence = conf

        # 3. Parse and Route
        match = self.sheet_regex.search(extracted_text.upper())
        if match:
            discipline = match.group(1)
            sheet_type = match.group(2)
            key = discipline + sheet_type
            
            expert = self.routing_map.get(key, self.routing_map.get(discipline, "manual_review"))
            
            return {
                "expert": expert,
                "detected_text": extracted_text,
                "parsed_sheet": match.group(0),
                "discipline": discipline,
                "sheet_type_code": sheet_type,
                "bbox": [x1, y1, x2, y2],
                "detection_confidence": float(scores[best_idx]),
                "ocr_confidence": float(ocr_confidence),
                "overall_confidence": float(scores[best_idx] * ocr_confidence)
            }
        
        return {
            "expert": "manual_review",
            "detected_text": extracted_text,
            "reason": "Regex mismatch on OCR output",
            "overall_confidence": float(ocr_confidence)
        }

# ── Test the Wrapper ──────────────────────────────────────────────────────────
# Load a sample image from the validation set to verify
sample_img_paths = list((DATASET_DIR / 'val' / 'images').glob('*.png'))
if not sample_img_paths:
    print("⚠️ No validation images found. Skipping inference test.")
else:
    sample_img_path = sample_img_paths[0]
    sample_img = cv2.imread(str(sample_img_path))

    router = RouterExpert(str(OUTPUT_DIR / "router-expert.onnx"), ocr_engine)
    result = router.predict(sample_img)

    print("\n🧪 Router Expert Inference Test:")
    print(json.dumps(result, indent=2))

# ── Generate MOSAIC Manifest ──────────────────────────────────────────────────
# ✅ FIX 3: Safe fallback for val_results in case Cell 3 was interrupted
try:
    map50 = float(val_results.box.map50)
    map50_95 = float(val_results.box.map)
except NameError:
    map50 = 0.0
    map50_95 = 0.0
    print("⚠️ Warning: 'val_results' not found. Using 0.0 for metrics. Ensure Cell 3 completed successfully.")

manifest = {
    "schema": "mosaic.expert-model.v1",
    "id": "router-expert",
    "name": "Sheet Router & Metadata Expert",
    "task": "object_detection_and_ocr",
    "num_classes": 1, 
    "classes": {0: "sheet_number"},
    "metrics": {
        "mAP50": map50,
        "mAP50_95": map50_95
    },
    "pipeline_stage": "ingest.router",
    "export": {"format": "onnx", "opset": 17, "imgsz": 640, "dynamic_batch": True},
    "dependencies": {"ocr_engine": "paddleocr-v4"}
}

with open(OUTPUT_DIR / "router-expert_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(f"\n✅ Manifest saved to {OUTPUT_DIR / 'router-expert_manifest.json'}")

📦 Exporting YOLO model to ONNX...
Ultralytics 8.4.190 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs

PyTorch: starting from '/kaggle/working/runs/router_expert/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (6.2 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 10 packages in 414ms
Prepared 1 package in 63ms
Installed 1 package in 7ms
 + onnxslim==0.1.98

requirements: AutoUpdate success ✅ 1.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 17...
ONNX: slimming with onnxslim 0.1.98...
ONNX: export success ✅ 4.3s, saved as '/k

/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
/tmp/ipykernel_49/2677118392.py:93: DeprecationWarning: Please use `predict` instead.
  ocr_result = self.ocr.ocr(crop, cls=True)


TypeError: PaddleOCR.predict() got an unexpected keyword argument 'cls'

In [ ]:
# CELL N-1: Evaluate & Export to ONNX
from ultralytics import YOLO
import shutil

best_model = YOLO(str(WORK_DIR / "runs" / "weights" / "best.pt"))
val_results = best_model.val(data=str(DATASET_DIR / "data.yaml"))

# Export with dynamic batch size and simplified graph
onnx_path = best_model.export(format="onnx", imgsz=1024, dynamic=True, simplify=True, opset=17)

# Copy to standardized output directory
OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
shutil.copy(onnx_path, OUTPUT_DIR / f"{EXPERT_NAME}-expert.onnx")

# CELL N: Generate MOSAIC Manifest
manifest = {
    "schema": "mosaic.expert-model.v1",
    "id": f"{EXPERT_NAME}-expert",
    "name": f"{EXPERT_NAME.title()} Domain Expert",
    "task": "object_detection" if "seg" not in str(best_model.task) else "instance_segmentation",
    "num_classes": len(CLASSES),
    "classes": {i: name for i, name in enumerate(CLASSES)},
    "metrics": {
        "mAP50": float(val_results.box.map50),
        "mAP50_95": float(val_results.box.map),
        "seg_mAP50": float(val_results.seg.map50) if hasattr(val_results, 'seg') else None
    },
    "pipeline_stage": f"ingest.{EXPERT_NAME}",
    "export": {"format": "onnx", "opset": 17, "imgsz": 1024, "dynamic_batch": True}
}

import json
with open(OUTPUT_DIR / f"{EXPERT_NAME}-expert_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
    
print(f"✅ {EXPERT_NAME} Expert exported to {OUTPUT_DIR}")

In [ ]:
import re

def route_sheet_page(ocr_text: str, image_tensor):
    # Regex for standard sheet numbering: [1-2 Letters][1 Digit][2 Digits]
    # e.g., A101, AS01, S203, M301, E401, P501
    match = re.search(r'\b([A-Z]{1,2})(\d)(\d{2})\b', ocr_text)
    
    if not match:
        return {"expert": "general_review", "confidence": 0.1}
        
    discipline = match.group(1)
    sheet_type = match.group(2)
    
    routing_map = {
        "G": "general", "A0": "general",
        "AS": "site_plan",
        "A1": "floor_plan", "A2": "floor_plan", # A2 can be plan or elevation, router checks layout aspect ratio
        "A3": "section", 
        "A4": "detail_schedule", "A5": "detail_schedule",
        "S": "structural",
        "M": "mep", "E": "mep", "P": "mep"
    }
    
    expert_name = routing_map.get(discipline + sheet_type, routing_map.get(discipline, "general_review"))
    
    return {
        "expert": expert_name,
        "discipline": discipline,
        "sheet_type_code": sheet_type,
        "confidence": 0.95 # From OCR confidence
    }

# Example Usage:
# route = route_sheet_page("PROJECT: TEST \n SHEET: A101", img)
# if route["expert"] == "floor_plan":
#     results = floor_plan_expert.predict(img)

## 
💡 Why This Architecture?
Separation of Concerns: YOLO is incredibly fast at finding where the text is, even if the paper is crumpled, rotated, or has low contrast. PaddleOCR is incredibly accurate at reading the text once it's isolated.
ONNX Compatibility: Exporting PaddleOCR into the same ONNX file as YOLO creates a massive, fragile graph that often fails in Node.js/C++ runtimes. Keeping YOLO as the ONNX model and wrapping it with a Python OCR post-processor is the industry standard for hybrid pipelines.
Regex Safety Net: The sheet_regex ensures that even if the OCR misreads "A101" as "A1O1" (letter O instead of zero), the routing logic can still be designed to handle common OCR typos, or at least flag it for the manual_review expert with high transparency.



### 📊 Training Strategy & Best Practices
Synthetic First, Real Second: Start training each expert on 10,000+ perfectly labeled synthetic images (generated by the scripts above). This teaches the model the ideal geometry. Then, fine-tune on 500–1,000 real images (CubiCasa5K, FloorPlanCAD, or scanned PDFs) to teach it to handle noise, skew, and poor scan quality.
Discipline-Specific Augmentation:
Architectural: Rotate 90/180/270 degrees (plans are often scanned upside down).
Structural: Add "salt and pepper" noise to simulate poor blueprint reproduction.
MEP: Use heavy Mosaic augmentation to teach the model to disentangle overlapping ducts and pipes.
Confidence Thresholding: Each expert must output a confidence score (0.0–1.0). The MOSAIC aggregator will route any detection with confidence < 0.65 to a human review queue, preventing silent failures.
ONNX Standardization: Exporting all models to ONNX with dynamic=True allows the MOSAIC C++/Node.js inference engine to batch process multiple sheets simultaneously on a single GPU, drastically reducing pipeline latency.
This MoE approach ensures that your Structural expert isn't confused by architectural furniture, and your MEP expert isn't distracted by exterior elevation hatches. Each model becomes a true master of its specific domain.


## Site Plan Expert (AS)
Goal: Detect property lines, setbacks, north arrows, grading contours, and site utilities.
Architecture: YOLOv8-seg (Instance Segmentation).

In [ ]:
# CELL 1: Site Plan Dataset Generation
def generate_site_dataset(n=1500):
    for split, count in [("train", int(n*0.8)), ("val", int(n*0.2))]:
        img_dir = DATASET_DIR / split / "images"
        lbl_dir = DATASET_DIR / split / "labels"
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)
        
        classes = {"property_line": 0, "setback": 1, "north_arrow": 2, "contour": 3}
        
        for i in tqdm(range(count)):
            img = Image.new("RGB", (1024, 1024), "white")
            draw = ImageDraw.Draw(img)
            labels = []
            
            # Property line (thick dashed)
            draw.rectangle([100, 100, 924, 924], outline="black", width=4)
            labels.append((0, [0.1, 0.1, 0.9, 0.1, 0.9, 0.9, 0.1, 0.9])) # Polygon normalized
            
            # North arrow
            draw.polygon([(500, 50), (480, 90), (520, 90)], fill="black")
            labels.append((2, [0.48, 0.05, 0.52, 0.05, 0.50, 0.09]))
            
            img.save(img_dir / f"{i:04d}.png")
            with open(lbl_dir / f"{i:04d}.txt", "w") as f:
                for cls, poly in labels:
                    f.write(f"{cls} " + " ".join(f"{p:.4f}" for p in poly) + "\n")

generate_site_dataset()
# CELL 2: Train YOLOv8s-seg
# model = YOLO('yolov8s-seg.pt')
# model.train(data=str(DATASET_DIR/'data.yaml'), epochs=100, imgsz=1024, task='segment')